# CS-320 (Artificial Neural Networks & Deep Learning) Project P03
## Densely Connected Convolutional Networks (DenseNet, Huang et al., CVPR 2017)

**Team Members & Individual Viva Ownership:**
- **Mohammad Abdul Rehman** (Team & Paper Lead): `_DenseLayer`, `_DenseBlock`, `DenseNet`, Hand Parameter Formulas, CIFAR-10+ Reproduction Benchmark. Ablations: Growth rate $k \in \{8, 12, 24\}$, Bottleneck on/off.
- **Muhammad Wajahat** (Data & Pipeline Lead): Custom Datasets & DataLoaders, Train-Split-Only Normalization, `_Transition` Layer, Classification Metrics. Ablations: Compression factor $\theta \in \{0.5, 1.0\}$, Input Resolution ($28, 64, 128$).
- **Aiman Aslam** (Experiments Lead): Training & Validation Engine (AMP, Checkpointing), Sanity Checks, LR Range Test, Clinical Calibration ($\ge 95\%$ Sensitivity), Error Analysis, Gradio Demonstration. Ablations: Dropout $p \in \{0.0, 0.2\}$, Threshold Calibration.

---

## Section 0: Environment Setup, Repository Sync & Dependency Installation
This cell handles Google Colab setup, clones/pulls the team repository (`wajahat2005/ANN-SEM-Project`), configures Python's module search path (`sys.path`), and installs dependencies.

In [ ]:
# 0.1 Mount Google Drive for persistent and resumable checkpointing
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running in Google Colab or Drive already mounted.")

# 0.2 Clone or Sync the Team Repository
REPO_URL = "https://github.com/wajahat2005/ANN-SEM-Project.git"
REPO_DIR = "/content/ANN-SEM-Project"

if IN_COLAB:
    if not os.path.exists(os.path.join(REPO_DIR, "src")):
        print(f"Cloning team repository from {REPO_URL}...")
        !rm -rf /content/ANN-SEM-Project
        !git clone https://github.com/wajahat2005/ANN-SEM-Project.git /content/ANN-SEM-Project
    else:
        print(f"Repository folder exists at {REPO_DIR}. Pulling latest changes...")
        !cd /content/ANN-SEM-Project && git pull || true

    # Navigate into repository root
    if os.path.exists(REPO_DIR):
        %cd /content/ANN-SEM-Project

# 0.3 Add working directory and repository root to sys.path to guarantee 'src' imports succeed
curr_dir = os.path.abspath(os.getcwd())
if curr_dir not in sys.path:
    sys.path.insert(0, curr_dir)
if IN_COLAB and os.path.exists(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"[Environment] Working Directory: {os.getcwd()}")
print(f"[Environment] sys.path[0]:       {sys.path[0]}")
assert os.path.exists("src"), "ERROR: 'src' directory not found! Please check repository path."

# 0.4 Install required dependencies
!pip install -q -r requirements.txt || pip install -q torch torchvision medmnist torchinfo gradio scikit-learn matplotlib pandas tqdm pyyaml

# Create directories for artifacts and checkpoints
os.makedirs("results", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

# 0.5 Verify GPU runtime and environment
import torch
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("Automatic Mixed Precision (AMP): Supported (torch.autocast)")
else:
    print("NOTE: Running on CPU. For fast training, enable GPU via: Runtime > Change runtime type > T4 GPU")

## Section 1: Data Verification, Pipeline Checks & Zero-Leakage Stats
- Strict separation of statistics: Mean and standard deviation are computed **strictly** on the 45,000 CIFAR-10 training split and the 4,708 PneumoniaMNIST training split.
- Data augmentation for CIFAR-10+: 4-pixel random crop and horizontal reflection flip.

In [ ]:
import os
from PIL import Image
from IPython.display import display

from src.data.cifar import get_cifar10_dataloaders, plot_cifar_sample_grid
from src.data.pneumonia import get_pneumonia_dataloaders, plot_pneumonia_sample_grid

# 1. Load CIFAR-10 DataLoaders
# Tip: Use smoke=False for the full 45k/5k/10k dataset, or smoke=True for a rapid offline check
cifar_train, cifar_val, cifar_test, cifar_meta = get_cifar10_dataloaders(batch_size=64, smoke=False)
plot_cifar_sample_grid(cifar_train, save_path="results/cifar_sample_grid.png")

# 2. Load PneumoniaMNIST DataLoaders (size=64)
pneumo_train, pneumo_val, pneumo_test, pneumo_meta = get_pneumonia_dataloaders(batch_size=64, size=64, smoke=False)
plot_pneumonia_sample_grid(pneumo_train, save_path="results/pneumonia_sample_grid.png")

# Display sample grids
if os.path.exists("results/cifar_sample_grid.png"):
    display(Image.open("results/cifar_sample_grid.png"))
if os.path.exists("results/pneumonia_sample_grid.png"):
    display(Image.open("results/pneumonia_sample_grid.png"))

## Section 2: Architecture Verification, Analytical Parameter Formulas & Shape Progression
- Hand formula verification: $\text{BN} = 2 \cdot C_{in}$, $\text{Conv3x3} = 9 \cdot C_{in} \cdot k$.
- Proving Block 1 Layer 1 ($C_{in}=16, k=12$) equals **exactly 1,760 parameters**.
- Verifying DenseNet-40 matches $\sim 1.02\text{M}$ and DenseNet-BC-100 matches $\sim 0.77\text{M}$.
- Verifying custom DenseNet-121 matches `torchvision.models.densenet121` parameter-for-parameter (7,978,856 params).

In [ ]:
import os
from src.utils.param_count import (
    verify_block1_layer1_params,
    verify_densenet40_params,
    verify_densenet_bc_100_params,
    compare_with_torchvision_densenet121,
    print_torchinfo_summary
)
from src.utils.shape_checks import generate_densenet40_shape_table
from src.models.densenet import densenet40_cifar

# 1. Run exact analytical parameter proofs
verify_block1_layer1_params()
verify_densenet40_params()
verify_densenet_bc_100_params()
compare_with_torchvision_densenet121()

# 2. Generate and display layer-by-layer shape table for DenseNet-40
shape_table_path = "results/shape_table.md"
generate_densenet40_shape_table(save_path=shape_table_path)
if os.path.exists(shape_table_path):
    with open(shape_table_path, "r", encoding="utf-8") as f:
        lines = f.readlines()[:25]
        print("".join(lines) + "\n... [See results/shape_table.md for all 47 layers]")

# 3. Print torchinfo summary table
model = densenet40_cifar()
print_torchinfo_summary(model, input_size=(1, 3, 32, 32))

## Section 3: Optimization Sanity Checks
1. **Initial Loss Check:** Confirms loss $\approx \ln(C) = \ln(10) \approx 2.3026$ for an untrained network on CIFAR-10.
2. **Gradient Flow Check:** Confirms non-zero gradients across 100% of trainable weight tensors.
3. **Single-Batch Overfitting:** Overfits a single mini-batch within 60 steps to ensure optimization capacity.

In [ ]:
from src.train.sanity import run_sanity_checks

sample_x, sample_y = next(iter(cifar_train))
sanity_results = run_sanity_checks(model, sample_x, sample_y, num_classes=10)

## Section 4: Learning Rate Range Test
- Exponential learning rate sweep from $10^{-6}$ to $2.0$ over 100 steps.
- Identifies point of steepest descent to validate base learning rate choice.

In [ ]:
import os
from PIL import Image
from IPython.display import display
import torch.nn as nn
from src.train.lr_range_test import run_lr_range_test
from src.train.engine import get_device

device = get_device()
criterion = nn.CrossEntropyLoss()
lrs, losses, suggested_lr = run_lr_range_test(
    model=model.to(device),
    dataloader=cifar_train,
    criterion=criterion,
    device=device,
    min_lr=1e-6,
    max_lr=2.0,
    num_iter=100,
    save_path="results/lr_range_test.png"
)

if os.path.exists("results/lr_range_test.png"):
    display(Image.open("results/lr_range_test.png"))

## Section 5: CIFAR-10+ Reproduction Benchmark
Trains and compares:
1. **DenseNet-40 ($k=12$)** (Paper error: 5.24%)
2. **DenseNet-BC-100 ($k=12$)** (Paper error: 4.51%)
3. **ResNet-56** (Paper baseline error: ~6.97%)

*Note:* On Colab GPU, set `--epochs 300` for full reproduction (~2.5 hours under AMP). Set `--smoke` for a fast 1-minute test.

In [ ]:
import os
from PIL import Image
from IPython.display import display

# Execute CIFAR-10 reproduction (change --smoke to --epochs 300 for the full Colab paper run)
!python -m experiments.reproduce_cifar --model densenet40 --smoke --batch-size 64 --lr 0.1

# Display training curves
for curve_file in [
    "results/cifar10_densenet40_300ep_curves.png",
    "results/cifar10_densenet40_50ep_curves.png",
    "results/cifar10_densenet40_smoke_curves.png",
]:
    if os.path.exists(curve_file):
        display(Image.open(curve_file))
        break

## Section 6: Medical Transfer Extension on PneumoniaMNIST (size=64)
Evaluates:
1. **Simple CNN** (5-layer baseline adapted from MedMNIST notebook)
2. **Custom DenseNet** (DenseNet-BC adapted for 1-channel 64x64)
3. **Fine-Tuned DenseNet-121** (Torchvision ImageNet pretrained)
- Class imbalance handled via class-weighted CrossEntropyLoss.

In [ ]:
import os
from PIL import Image
from IPython.display import display

# Execute PneumoniaMNIST benchmark (change --smoke to --epochs 40 for full GPU training)
!python -m experiments.run_pneumonia --model custom_densenet --size 64 --smoke

# Display calibration and evaluation figures
if os.path.exists("results"):
    for fig_name in sorted(os.listdir("results")):
        if fig_name.startswith("pneumonia_") and fig_name.endswith(".png"):
            print(f"Artifact: {fig_name}")
            display(Image.open(os.path.join("results", fig_name)))

## Section 7: Systematic Ablation Studies (One-Factor-At-A-Time)
Evaluates each factor from the fixed baseline (`configs/baseline.yaml`):
- Growth rate $k \in \{8, 12, 24\}$
- Bottleneck on vs off
- Compression $\theta \in \{0.5, 1.0\}$
- Input resolution ($28, 64, 128$)
- Dropout $p \in \{0.0, 0.2\}$

In [ ]:
import os
import pandas as pd
from PIL import Image
from IPython.display import display

# Run ablation suite (change --smoke to --epochs 20 for full Colab exploration)
!python -m experiments.run_ablations --factor growth_rate --smoke

csv_path = "results/ablations.csv"
if os.path.exists(csv_path):
    df_ablations = pd.read_csv(csv_path)
    display(df_ablations)

# Display ablation factor plots
for f in ["growth_rate", "bottleneck", "compression", "resolution", "dropout"]:
    p = f"results/ablation_{f}.png"
    if os.path.exists(p):
        display(Image.open(p))

## Section 8: Diagnostic Error Analysis & Misclassifications
- Detailed confusion matrix and per-class error breakdown.
- Visual inspection grid of misclassified test samples with model confidence.

In [ ]:
import os
from PIL import Image
from IPython.display import display
from src.eval.error_analysis import analyze_misclassifications, plot_misclassified_grid

misclass_data = analyze_misclassifications(
    model=model.to(device),
    dataloader=cifar_test,
    device=device,
    class_names=cifar_meta["classes"],
    max_samples=16
)
plot_misclassified_grid(
    misclass_data,
    class_names=cifar_meta["classes"],
    mean=cifar_meta["mean"],
    std=cifar_meta["std"],
    save_path="results/misclassified_grid.png"
)
if os.path.exists("results/misclassified_grid.png"):
    display(Image.open("results/misclassified_grid.png"))

## Section 9: Clinical Calibration & Threshold Tuning
- Selecting threshold on validation set for $\ge 95\%$ sensitivity.
- Reliability diagram and Expected Calibration Error (ECE).
- Post-hoc temperature scaling.

In [ ]:
import os
from PIL import Image
from IPython.display import display

# Display calibration artifacts generated during PneumoniaMNIST experiments
if os.path.exists("results"):
    for f in ["reliability.png", "tradeoff.png", "cm.png"]:
        for fname in sorted(os.listdir("results")):
            if f in fname:
                print(f"Artifact: {fname}")
                display(Image.open(os.path.join("results", fname)))

## Section 10: Interactive Gradio Diagnostic Explorer Launch
- Launches interactive diagnostic explorer with `share=True` for a public Colab URL.
- Features: Dataset selector, Sensitivity slider, and Figure 5 Feature-Reuse Heatmap / Grad-CAM.

In [ ]:
from app.gradio_demo import build_app

demo_app = build_app()
# Launch with share=True to obtain a public URL on Google Colab
demo_app.launch(share=True)